In [5]:
import os

from pyspark.sql import SparkSession

POSTGRES_JAR = os.environ.get(
    "POSTGRES_JDBC_JAR", "/opt/spark-jars/postgresql-42.7.4.jar"
)

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .appName("spell-classifiers")
    .config("spark.jars", POSTGRES_JAR)
    .getOrCreate()
)

df = (
    spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://host.docker.internal:5432/postgres")
    .option("dbtable", "spells")
    .option("user", "postgres")
    .option("password", "password")
    .option("driver", "org.postgresql.Driver")
    .load()
)

df.show()

26/05/20 01:57:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+---+------+---------------+---------+---------+------------+---------------+---------+
| id|damage|    description|image_url|mana_cost|        name|play_effect_url|sound_url|
+---+------+---------------+---------+---------+------------+---------------+---------+
|  1|     6|Наносит 6 урона|     NULL|        4|Огненный шар|           NULL|     NULL|
|  2|     6|Наносит 6 урона|     NULL|        1|      Молния|           NULL|     NULL|
+---+------+---------------+---------+---------+------------+---------------+---------+



In [4]:
import numpy as np

LABEL_FIREBALL = 0
LABEL_LIGHTNING = 1
CLASS_NAMES = {LABEL_FIREBALL: "Огненный шар", LABEL_LIGHTNING: "Молния"}

# Реальные строки из Spark (без pandas)
rows = df.select("mana_cost", "damage", "name").collect()

X_real, y_real, names_real = [], [], []
for row in rows:
    X_real.append([float(row.mana_cost), float(row.damage)])
    if row.name.strip() == "Молния":
        y_real.append(LABEL_LIGHTNING)
    else:
        y_real.append(LABEL_FIREBALL)
    names_real.append(row.name)

X_real = np.array(X_real)
y_real = np.array(y_real)
print("Из БД:")
for name, (mana, dmg), label in zip(names_real, X_real, y_real):
    print(f"  {name}: mana={mana:.0f}, damage={dmg:.0f} -> {CLASS_NAMES[label]}")

Из БД:
  Огненный шар: mana=4, damage=6 -> Огненный шар
  Молния: mana=1, damage=6 -> Молния
